# Environment check
This notebook trains the checked-in probabilistic ensemble; it does not copy model code.

In [ ]:
import platform
import subprocess
import sys

import torch

print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Torch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}')
subprocess.run(['nvidia-smi'], check=False)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before training.'

# Repository install
Set the repository URL to the branch or release you want to train.

In [ ]:
from pathlib import Path

REPOSITORY_URL = 'https://github.com/your-organization/imbalance-pipeline.git'  # replace with your fork
PROJECT = Path('/content/imbalance-pipeline')
!git clone --depth=1 $REPOSITORY_URL $PROJECT
%cd $PROJECT
!pip install --quiet '.[ml]'

# Dataset acquisition/upload
Use a point-in-time dataset export from this pipeline. Never upload secrets or a ClickHouse password to the notebook.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DATASET_PATH = Path('/content/drive/MyDrive/imbalance/dataset')
assert (DATASET_PATH / 'metadata.json').is_file(), 'Copy an exported dataset directory to Google Drive first.'

# Configuration
The validation, calibration and untouched test partitions are chronological and purged by the package.

In [ ]:
from imbalance_pipeline.training.train import TrainingConfig

CANDIDATE_DIR = Path('/content/drive/MyDrive/imbalance/candidates/imbalance-colab')
config = TrainingConfig(
    seeds=(17, 29, 43),
    device='cuda',
    mixed_precision=True,
    deterministic=True,
)

# GPU training
Training fits preprocessing only on the train period and writes three ONNX ensemble members.

In [ ]:
from imbalance_pipeline.training.train import train_ensemble

candidate = train_ensemble(DATASET_PATH, CANDIDATE_DIR, config)
print(candidate)

# Evaluation display
A candidate must pass the untouched-test promotion gate before it is treated as deployable.

In [ ]:
import json
import pandas as pd

from imbalance_pipeline.training.metrics import EvaluationReport, promotion_decision

evaluation = json.loads((candidate / 'evaluation.json').read_text())['candidate']
baselines = json.loads((candidate / 'baseline_evaluation.json').read_text())
pd.DataFrame({'candidate': evaluation, **{name: report for name, report in baselines.items()}}).T
candidate_report = EvaluationReport(**evaluation)
decision = promotion_decision(
    candidate_report,
    EvaluationReport(**baselines['persistence']),
    EvaluationReport(**baselines['classical']),
)
print(decision)
assert decision.promote, f'Rejected candidate: {decision.reasons}'

# ONNX export and validation
`train_ensemble` already exports the members; validate checksums and the serving contract again before transfer.

In [ ]:
from imbalance_pipeline.model.bundle import ModelManifest, validate_bundle

manifest = ModelManifest.load(candidate)
validation = validate_bundle(candidate, expected_schema_hash=manifest.feature_schema_hash)
assert validation.valid, validation.reason
print({'model_version': manifest.model_version, 'schema': manifest.feature_schema_hash})

# Bundle download
Download only the checksummed candidate bundle. Promotion to production stays an operator action in the Docker environment.

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(str(candidate), 'zip', root_dir=candidate)
files.download(archive)